# Bias Mitigation — Reweighing (pre-processing)

**AIF360 class:** `aif360.algorithms.preprocessing.Reweighing` &nbsp;|&nbsp; requires `pip install -e .[mitigation]`

Reweighing assigns per-instance weights so the protected attribute and label become statistically independent; the weights are passed to the classifier as `sample_weight`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from fair_ed.mitigation.base import (
    load_splits, group_dicts, to_aif360_dataset,
    dataset_from_predictions, evaluate_subgroups,
)
from fair_ed.modeling import prepare_xy, standardize, FEATURE_COLUMNS

# --- Prediction task and binary protected attribute (edit to taste) ---
OUTCOME = "outcome_hospitalization"   # e.g. outcome_critical, outcome_sepsis, outcome_pe, ...
PROTECTED = "sex_priv"                 # 1 = male (privileged), 0 = female (unprivileged)

df_train, df_test = load_splits()
for _df in (df_train, df_test):
    _df[PROTECTED] = (_df["gender"].astype(str).str.upper().str[0] == "M").astype(int)

X_train, X_test, y_train, y_test = prepare_xy(df_train, df_test, OUTCOME)
X_train, X_test, _ = standardize(X_train, X_test)
FEAT = list(X_train.columns)

train_frame = X_train.assign(**{PROTECTED: df_train[PROTECTED].to_numpy(), OUTCOME: y_train.to_numpy()})
test_frame = X_test.assign(**{PROTECTED: df_test[PROTECTED].to_numpy(), OUTCOME: y_test.to_numpy()})

priv, unpriv = group_dicts(PROTECTED)

def subgroup_labels(ds):
    return np.where(ds.protected_attributes.ravel() == 1, "Male", "Female")

## Apply Reweighing and evaluate subgroups

In [ ]:
from fair_ed.mitigation.preprocessing import reweigh

train_ds = to_aif360_dataset(train_frame, FEAT, OUTCOME, PROTECTED)
test_ds = to_aif360_dataset(test_frame, FEAT, OUTCOME, PROTECTED)

# Reweighing returns per-instance weights to pass as sample_weight.
rw_ds = reweigh(train_ds, priv, unpriv)

clf = LogisticRegression(max_iter=1000)
clf.fit(rw_ds.features, rw_ds.labels.ravel(), sample_weight=rw_ds.instance_weights)

y_score = clf.predict_proba(test_ds.features)[:, 1]
y_pred = (y_score >= 0.5).astype(int)

ci = evaluate_subgroups(test_ds.labels.ravel(), y_pred, y_score, subgroup_labels(test_ds), B=200)
print(ci.pivot(index="Group", columns="Metric", values="Mean").round(3))